# DarkSight — E2 and E3 on Kaggle

**The question:** does brightening a dark photo before detection make detection better?

**E2 (baseline)** — train YOLOv8 on the raw dark images, test on raw dark images.
**E3** — take that *same* trained model and test it on Zero-DCE brightened images.

One training run answers both. Every step below is written out in plain Python so you can read it top to bottom.

### Settings (right-hand panel)
**Accelerator: GPU T4 x2** · **Internet: On** · **Persistence: Files and Variables** → then **Run All** (~1.5–2 h).

### The plan
| Step | What happens |
|---|---|
| 1 | Install packages, download ExDark (7,363 dark photos with boxes drawn by hand) |
| 2 | Convert the boxes to the format YOLO expects |
| 3 | Brighten the test photos with Zero-DCE |
| 4 | Train YOLOv8 on the raw training photos |
| 5 | Test that model twice: on raw photos (E2) and on brightened photos (E3) |
| 6 | Compare the two scores and look at example pictures |

## Step 0 · Setup

In [ ]:
!pip -q install ultralytics gdown

import os, shutil, time
from pathlib import Path
import torch

START = time.time()
WORK = Path('/kaggle/working')      # kept after the session ends
TEMP = Path('/kaggle/temp')         # scratch space, deleted at the end
DEVICE = [0, 1]                     # both T4 GPUs. If training errors, change to 0 for a single GPU.

print('GPUs available:', torch.cuda.device_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# The 12 object classes, in the order the ExDark authors numbered them.
# YOLO needs numbers, so Bicycle=0, Boat=1, ... Table=11.
CLASSES = ["Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
           "Chair", "Cup", "Dog", "Motorbike", "People", "Table"]
CLASS_ID = {name.lower(): i for i, name in enumerate(CLASSES)}
MAX_SIDE = 1280        # shrink very large photos so they fit in GPU memory

# Two small files from our repo: the official train/val/test list, and the Zero-DCE weights.
!git clone -q --depth 1 https://github.com/Manideep-21/darksight.git {TEMP}/repo
SPLIT_FILE   = TEMP / 'repo/configs/exdark_imageclasslist.txt'
ZERODCE_FILE = TEMP / 'repo/weights/zerodce_epoch99.pth'
print(SPLIT_FILE.exists(), ZERODCE_FILE.exists())

## Step 1 · Download ExDark

Two files from the authors' Google Drive: the photos (1.5 GB) and the hand-drawn boxes (5 MB).

In [ ]:
SRC = TEMP / 'exdark_src'
if not SRC.exists():
    !gdown -q 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O {TEMP}/images.zip
    !gdown -q 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O {TEMP}/boxes.zip
    !mkdir -p {SRC} && unzip -q -o {TEMP}/images.zip -d {SRC} && unzip -q -o {TEMP}/boxes.zip -d {SRC}
    !rm -rf {SRC}/__MACOSX

# The photos sit in 12 folders (one per class); the box files mirror them.
IMAGE_DIR = SRC / 'ExDark'
BOX_DIR   = SRC / 'ExDark_Annno'
print(sorted(p.name for p in IMAGE_DIR.iterdir() if p.is_dir()))
print('photos:', sum(1 for p in IMAGE_DIR.rglob('*') if p.is_file()))

In [ ]:
# A photo is called 2015_00001.png and its boxes are in 2015_00001.png.txt,
# but the spelling of the extension is not always consistent (.jpg / .JPG / .JPEG).
# So we look files up by their name WITHOUT the extension.
photos = {p.stem.lower(): p for p in IMAGE_DIR.rglob('*') if p.is_file() and not p.name.startswith('.')}
boxes  = {p.name.lower().removesuffix('.txt').rsplit('.', 1)[0]: p for p in BOX_DIR.rglob('*.txt')}
print(len(photos), 'photos,', len(boxes), 'box files')

# What one box file looks like: a header, then "class left top width height" in pixels.
print(open(boxes['2015_00001']).read()[:120])

## Step 2 · Convert the boxes to YOLO format

ExDark stores a box as **left, top, width, height in pixels**.
YOLO wants **centre-x, centre-y, width, height as fractions of the image size**:

```
x_centre = (left + width/2) / image_width        # 0.0 = left edge, 1.0 = right edge
```

Using fractions means the numbers stay correct even after we resize the photo.

In [ ]:
from PIL import Image

def convert_one(photo_path, box_path, out_photo, out_label):
    """Save one photo as PNG, and its boxes as one YOLO label file."""
    img = Image.open(photo_path).convert('RGB')     # some photos are greyscale
    W, H = img.size

    lines = []
    for line in Path(box_path).read_text(errors='ignore').splitlines():
        parts = line.split()
        if not parts or parts[0].startswith('%'):
            continue                                 # skip the header line
        name, l, t, w, h = parts[0], *map(float, parts[1:5])
        cls = CLASS_ID.get(name.lower())
        if cls is None:
            continue
        x1, y1 = max(l, 0), max(t, 0)                # a few boxes stick out past
        x2, y2 = min(l + w, W), min(t + h, H)        # the edge, so trim them
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue                                 # ignore slivers
        lines.append(f"{cls} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")

    if max(W, H) > MAX_SIDE:
        img.thumbnail((MAX_SIDE, MAX_SIDE))
    out_photo.parent.mkdir(parents=True, exist_ok=True)
    out_label.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_photo)
    out_label.write_text("\n".join(lines))
    return len(lines)

In [ ]:
# The ExDark authors fixed which photos are for training / validating / testing.
# Their file lists: name, class, lighting type, indoor-outdoor, split (1=train, 2=val, 3=test).
SPLIT_NAME = {'1': 'train', '2': 'val', '3': 'test'}

split_of, lighting_of = {}, {}
LIGHTING = {'1': 'Low', '2': 'Ambient', '3': 'Object', '4': 'Single', '5': 'Weak',
            '6': 'Strong', '7': 'Screen', '8': 'Window', '9': 'Shadow', '10': 'Twilight'}
for line in SPLIT_FILE.read_text().splitlines():
    c = line.split()
    if len(c) == 5 and c[1].isdigit():
        key = Path(c[0]).stem.lower()
        split_of[key] = SPLIT_NAME[c[4]]
        lighting_of[key] = LIGHTING[c[2]]

from collections import Counter
print(Counter(split_of.values()))   # expect 3000 train, 1800 val, 2563 test

In [ ]:
# Convert all 7,363 photos. Takes about 8 minutes.
RAW = TEMP / 'data/raw'
total_boxes = 0
for i, (key, photo) in enumerate(sorted(photos.items())):
    split = split_of[key]
    total_boxes += convert_one(photo, boxes[key],
                               RAW / 'images' / split / f'{key}.png',
                               RAW / 'labels' / split / f'{key}.txt')
    if i % 1000 == 0:
        print(f'{i}/{len(photos)} photos, {total_boxes} boxes so far')
print('done:', len(photos), 'photos,', total_boxes, 'boxes')

### Check the boxes landed in the right place — if this looks wrong, stop here

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt

def draw_boxes(img, boxes_xyxy, labels):
    """Draw rectangles + names on a copy of the image."""
    out = img.copy()
    for (x1, y1, x2, y2), text in zip(boxes_xyxy, labels):
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(out, text, (int(x1), max(int(y1) - 5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return out

def show_ground_truth(key, split='train'):
    img = np.array(Image.open(RAW / 'images' / split / f'{key}.png'))
    H, W = img.shape[:2]
    xyxy, names = [], []
    for line in (RAW / 'labels' / split / f'{key}.txt').read_text().splitlines():
        c, xc, yc, w, h = line.split()
        xc, yc, w, h = float(xc) * W, float(yc) * H, float(w) * W, float(h) * H
        xyxy.append((xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
        names.append(CLASSES[int(c)])
    return draw_boxes(np.clip(img * 2.0, 0, 255).astype('uint8'), xyxy, names)  # x2 brightness to see it

sample = [k for k in sorted(split_of) if split_of[k] == 'train'][:3]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key in zip(axes, sample):
    ax.imshow(show_ground_truth(key)); ax.set_title(key); ax.axis('off')
plt.show()

## Step 3 · Brighten the test photos with Zero-DCE

Zero-DCE is a tiny network (79,000 numbers). It does not output a brightened photo directly —
it outputs a **brightening curve** for each pixel, applied 8 times in a row:

```
new = old + a * old * (1 - old)
```

`a` comes from the network. Because `old * (1 - old)` is largest for mid-grey pixels and near zero
for black and white ones, dark areas get lifted while highlights stay put.

In [ ]:
import torch.nn as nn

class ZeroDCE(nn.Module):
    """7 small conv layers that predict 8 brightening curves (3 colours x 8 = 24 outputs)."""
    def __init__(self, f=32):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(3,     f, 3, 1, 1)
        self.e_conv2 = nn.Conv2d(f,     f, 3, 1, 1)
        self.e_conv3 = nn.Conv2d(f,     f, 3, 1, 1)
        self.e_conv4 = nn.Conv2d(f,     f, 3, 1, 1)
        self.e_conv5 = nn.Conv2d(f * 2, f, 3, 1, 1)     # takes conv3 + conv4 joined together
        self.e_conv6 = nn.Conv2d(f * 2, f, 3, 1, 1)
        self.e_conv7 = nn.Conv2d(f * 2, 24, 3, 1, 1)

    def forward(self, x):
        x1 = self.relu(self.e_conv1(x))
        x2 = self.relu(self.e_conv2(x1))
        x3 = self.relu(self.e_conv3(x2))
        x4 = self.relu(self.e_conv4(x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1)))
        curves = torch.tanh(self.e_conv7(torch.cat([x1, x6], 1)))
        for r in torch.split(curves, 3, dim=1):          # apply the 8 curves one after another
            x = x + r * (x * x - x)
        return x

zerodce = ZeroDCE().cuda().eval()
zerodce.load_state_dict(torch.load(ZERODCE_FILE, map_location='cuda'))
print(sum(p.numel() for p in zerodce.parameters()), 'parameters')

In [ ]:
@torch.no_grad()
def brighten(img):
    """numpy image (0-255) -> brightened numpy image (0-255)."""
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
    out = zerodce(x).clamp(0, 1)
    return (out[0].permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')

# before / after on one dark test photo
test_keys = sorted(k for k in split_of if split_of[k] == 'test')
demo = np.array(Image.open(RAW / 'images/test' / f'{test_keys[0]}.png'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(demo);            axes[0].set_title('raw');       axes[0].axis('off')
axes[1].imshow(brighten(demo));  axes[1].set_title('Zero-DCE');  axes[1].axis('off')
plt.show()

In [ ]:
# Brighten every test photo. The labels are copied unchanged: brightening moves no object,
# so the boxes are still correct. Takes about 3 minutes.
ENH = TEMP / 'data/enhanced'
shutil.rmtree(ENH, ignore_errors=True)
shutil.copytree(RAW / 'labels/test', ENH / 'labels/test')
(ENH / 'images/test').mkdir(parents=True, exist_ok=True)

for i, key in enumerate(test_keys):
    img = np.array(Image.open(RAW / 'images/test' / f'{key}.png'))
    Image.fromarray(brighten(img)).save(ENH / 'images/test' / f'{key}.png')
    if i % 500 == 0:
        print(f'{i}/{len(test_keys)}')
print('brightened', len(test_keys), 'test photos')

## Step 4 · Train YOLOv8 on the raw photos

A `.yaml` file tells YOLO where the photos are and what the classes are called.
We write two: one pointing at raw test photos (for E2), one at brightened test photos (for E3).
**Only the test folder differs** — same model, same labels — which is what makes the comparison fair.

In [ ]:
def write_yaml(path, train_dir, val_dir, test_dir):
    names = "\n".join(f'  {i}: {n}' for i, n in enumerate(CLASSES))
    Path(path).write_text(
        f"path: /\ntrain: {train_dir}\nval: {val_dir}\ntest: {test_dir}\nnames:\n{names}\n")
    return path

yaml_raw = write_yaml(TEMP / 'exdark_raw.yaml',
                      RAW / 'images/train', RAW / 'images/val', RAW / 'images/test')
yaml_enh = write_yaml(TEMP / 'exdark_enhanced.yaml',
                      RAW / 'images/train', RAW / 'images/val', ENH / 'images/test')
print(open(yaml_raw).read())

In [ ]:
from ultralytics import YOLO

# Start from a model already trained on COCO (ordinary daylight photos) and continue
# training it on ExDark. 'Fine-tuning' = it already knows what a car looks like,
# it just has to learn what a car looks like in the dark.
model = YOLO('yolov8s.pt')
model.train(
    data=str(yaml_raw),
    epochs=60,          # 60 passes over the 3000 training photos
    imgsz=640,          # photos are resized to 640x640 for the network
    batch=32,           # 32 photos at a time (16 per GPU)
    device=DEVICE,
    seed=0,             # same starting point every run, so results are repeatable
    patience=15,        # stop early if 15 epochs bring no improvement
    save_period=5,      # save a checkpoint every 5 epochs in case the session dies
    project=str(WORK / 'runs'), name='det_raw', exist_ok=True,
)
best = WORK / 'runs/det_raw/weights/best.pt'
shutil.copy(best, WORK / 'det_raw.pt')
print('trained model saved to', WORK / 'det_raw.pt')

*If the session died during training:* re-run every cell above (about 15 minutes), then instead of the
training cell run `YOLO('/kaggle/working/runs/det_raw/weights/last.pt').train(resume=True)`.

## Step 5 · Test the model twice

`model.val()` runs the model over the test photos, compares its boxes with the hand-drawn ones,
and reports **mAP@0.5**: for a box to count as correct it must overlap the true box by at least 50%
and have the right class. Higher is better; 1.0 would be perfect.

In [ ]:
model = YOLO(WORK / 'det_raw.pt')

print('=== E2: tested on RAW photos ===')
e2 = model.val(data=str(yaml_raw), split='test', imgsz=640, device=0)

print('=== E3: tested on BRIGHTENED photos ===')
e3 = model.val(data=str(yaml_enh), split='test', imgsz=640, device=0)

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'E2 — raw photos':        [e2.box.map50, e2.box.map, e2.box.mp, e2.box.mr],
    'E3 — brightened photos': [e3.box.map50, e3.box.map, e3.box.mp, e3.box.mr],
}, index=['mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall']).round(4)

difference = e3.box.map50 - e2.box.map50
print(summary.to_string())
print(f'\nBrightening changed mAP@0.5 by {difference:+.4f}')
print('Brightening HELPED' if difference > 0 else 'Brightening HURT the detector')
summary.to_csv(WORK / 'summary_E2_E3.csv')

In [ ]:
# Which classes changed the most?
per_class = pd.DataFrame({
    'E2': [e2.box.ap50[list(e2.box.ap_class_index).index(i)] if i in e2.box.ap_class_index else 0 for i in range(12)],
    'E3': [e3.box.ap50[list(e3.box.ap_class_index).index(i)] if i in e3.box.ap_class_index else 0 for i in range(12)],
}, index=CLASSES)
per_class['change'] = (per_class.E3 - per_class.E2).round(4)
print(per_class.round(4).sort_values('change').to_string())
per_class.to_csv(WORK / 'per_class_E2_E3.csv')

per_class[['E2', 'E3']].plot.bar(figsize=(11, 4), title='AP@0.5 per class: raw vs brightened')
plt.ylabel('AP@0.5'); plt.tight_layout(); plt.savefig(WORK / 'per_class_E2_E3.png', dpi=150); plt.show()

### Does the lighting type matter?
ExDark labels each photo with how it was lit. If brightening helps anywhere, it should help
the darkest ones ("Low", "Weak").

In [ ]:
rows = []
for light in ['Low', 'Weak', 'Ambient', 'Single', 'Strong', 'Twilight']:
    keys = [k for k in test_keys if lighting_of[k] == light]
    if len(keys) < 20:
        continue
    subset = TEMP / f'subset_{light}.txt'
    subset.write_text("\n".join(str(RAW / 'images/test' / f'{k}.png') for k in keys))
    subset_enh = TEMP / f'subset_{light}_enh.txt'
    subset_enh.write_text("\n".join(str(ENH / 'images/test' / f'{k}.png') for k in keys))
    r = model.val(data=str(write_yaml(TEMP / f'y_{light}.yaml', RAW/'images/train', RAW/'images/val', subset)),
                  split='test', imgsz=640, device=0, verbose=False)
    e = model.val(data=str(write_yaml(TEMP / f'y_{light}_e.yaml', RAW/'images/train', RAW/'images/val', subset_enh)),
                  split='test', imgsz=640, device=0, verbose=False)
    rows.append({'lighting': light, 'photos': len(keys),
                 'E2 raw': round(r.box.map50, 4), 'E3 brightened': round(e.box.map50, 4)})

by_light = pd.DataFrame(rows)
by_light['change'] = (by_light['E3 brightened'] - by_light['E2 raw']).round(4)
print(by_light.to_string(index=False))
by_light.to_csv(WORK / 'by_lighting_E2_E3.csv', index=False)

## Step 6 · Look at actual pictures

Numbers alone will not convince anyone. Here are the model's own detections, raw on the left,
brightened on the right, for a few test photos.

In [ ]:
def detect_and_draw(image_path):
    img = np.array(Image.open(image_path))
    result = model.predict(img[:, :, ::-1], conf=0.25, device=0, verbose=False)[0]  # YOLO expects BGR
    names = [f'{CLASSES[int(c)]} {s:.2f}' for c, s in zip(result.boxes.cls, result.boxes.conf)]
    return draw_boxes(img, result.boxes.xyxy.cpu().numpy(), names), len(names)

for key in test_keys[:4]:
    left,  n_raw = detect_and_draw(RAW / 'images/test' / f'{key}.png')
    right, n_enh = detect_and_draw(ENH / 'images/test' / f'{key}.png')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(left);  axes[0].set_title(f'{key} — raw: {n_raw} detections');        axes[0].axis('off')
    axes[1].imshow(right); axes[1].set_title(f'brightened: {n_enh} detections');          axes[1].axis('off')
    plt.tight_layout(); plt.savefig(WORK / f'example_{key}.png', dpi=120); plt.show()

In [ ]:
print('Files saved in /kaggle/working (download them from the Output panel on the right):')
for f in sorted(WORK.glob('*')):
    if f.is_file():
        print(f'  {f.name:32s} {f.stat().st_size/1e6:8.2f} MB')
print(f'\nTotal notebook time: {(time.time()-START)/60:.0f} minutes')
print('''
What to say about this experiment:
  E2 = detector trained on dark photos, tested on dark photos      (the baseline)
  E3 = the same detector, tested on brightened photos              (brighten at test time only)
  If E3 < E2, the detector was trained on dark photos, so brightened ones look
  unfamiliar to it - which is exactly why the next step trains a second detector
  ON brightened photos (that experiment is called E4).
''')